In [ ]:
import sys
!{sys.executable} -m pip install python-dotenv

In [1]:
"""
Test Job.get_next_request() integration with LLM_API.

This script verifies that the chat_history format returned by Job.get_next_request()
is compatible with LLM_API.process_request() method. It can be converted to a 
Jupyter notebook for interactive testing.
"""

import dbzero as db0
from collections import namedtuple
from statek.agent import Agent
from statek.executors.job import Job, JobDef, JobStatus
from statek.executors.chat_log_item import ChatLogItem
from statek.pyenv import PyEnv
from statek.llm_api import LLM_API, OpenRouter_API
from statek.settings import get_provider_settings
from statek.executors.utils import exec_step


In [2]:
# Initialize dbzero
db0.init(".dbzero_data")
db0.open("test-prefix")

In [3]:
# sample tools mocs

def add(a: int, b: int) -> int:
    """Adds two elements"""
    return a + b

def multiply(a: int, b: int) -> int:
    """multiply two elements"""
    return a * b

In [6]:

"""Create a Job instance"""
# Create agent and pyenv
agent = Agent(_system_prompt="""You are a helpful assistant. You solves given equations, 
but can only calculate one thin at a time. Need to use provided tools: \n{tools} \nYou only return one tool call at time e. g sum(2,5). Send only operation wrapped in print().
No comments just for eg: print(add(1,2)).""", _tools=[add,multiply], role="test")
pyenv = PyEnv(local_state={
    "multiply":multiply,
    "add":add
})


# Create job definition and job
job_def = JobDef(
    agent=agent,
    description="Solve this problem: {goal}",
    goal="2 + 3 * 5 + 4 * 2",
    warmup_code=None
)
job = Job(
    job_def=job_def,
    model_family="test",
    model="openai/gpt-4o",
    job_status=JobStatus.READY,
    py_env=pyenv
)


### Test 1: Verify chat_history Format Compatibility with LLM_API._build_messages

#### This test verifies that the generator returned by `get_next_request()` can be consumed by `LLM_API` without errors, ensuring proper formatting

In [7]:
# Get the request parameters
request = job.get_next_request()

print(f"\nRequest prompt: {request['prompt']}")
print(f"System prompt: {request['system_prompt']}")
for history in request['chat_history']:
    print(history)


Request prompt: Solve this problem: 2 + 3 * 5 + 4 * 2
System prompt: You are a helpful assistant. You solves given equations, 
but can only calculate one thin at a time. Need to use provided tools: 
>def add(a: int, b: int) -> int
>def multiply(a: int, b: int) -> int 
You only return one tool call at time e. g sum(2,5). Send only operation wrapped in print().
No comments just for eg: print(add(1,2)).


In [8]:
from dotenv import load_dotenv
# Create a test OpenRouter_API instance (won't make actual API call)
load_dotenv("./.env")
settings = get_provider_settings("OPENROUTER")
llm_api = OpenRouter_API(settings=settings)
response = await llm_api.process_request(**request)
job.append_chat_log(request, response.text)
print(response)

LLM_Response(text='print(multiply(3,5))', session_id=None)


In [9]:
# execute step
result = await exec_step(response.text, job)
print(result)


True


In [10]:
# Get the next request parameters
request = job.get_next_request()

print(f"\nRequest prompt: {request['prompt']}")
print(f"System prompt: {request['system_prompt']}")
print("History: ")
for history in request['chat_history']:
    print(history)


Request prompt: > 15

System prompt: You are a helpful assistant. You solves given equations, 
but can only calculate one thin at a time. Need to use provided tools: 
>def add(a: int, b: int) -> int
>def multiply(a: int, b: int) -> int 
You only return one tool call at time e. g sum(2,5). Send only operation wrapped in print().
No comments just for eg: print(add(1,2)).
History: 
Solve this problem: 2 + 3 * 5 + 4 * 2
print(multiply(3,5))


In [11]:
# execute step
response = await llm_api.process_request(**request)
job.append_chat_log(request, response.text)
print(response)

LLM_Response(text='print(multiply(3,5))', session_id=None)


In [12]:
# Model doesn't give proper answers, but rigthn now its not case of test. Just try execute next step and check history generation
result = await exec_step(response.text, job)
print(result)

True


In [13]:
# Get the next request parameters
request = job.get_next_request()

print(f"\nRequest prompt: {request['prompt']}")
print(f"System prompt: {request['system_prompt']}")
print("History: ")
for history in request['chat_history']:
    print(history)


Request prompt: > 15

System prompt: You are a helpful assistant. You solves given equations, 
but can only calculate one thin at a time. Need to use provided tools: 
>def add(a: int, b: int) -> int
>def multiply(a: int, b: int) -> int 
You only return one tool call at time e. g sum(2,5). Send only operation wrapped in print().
No comments just for eg: print(add(1,2)).
History: 
Solve this problem: 2 + 3 * 5 + 4 * 2
print(multiply(3,5))
> 15

print(multiply(3,5))
